In [17]:
import pandas as pd 
import requests
import re
from bs4 import BeautifulSoup
from PyPDF2 import PdfReader
from io import BytesIO
import time

In [18]:
session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/pdf,application/xhtml+xml"
}

base_url = "https://www.supremecourt.gov/opinions/slipopinion/"

cases = []

In [19]:
for term in range(18, 24):

    url = f"{base_url}{term}"
    print("Scraping:", url)

    r = session.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    for link in soup.find_all("a", href=True):

        href = link["href"]

        if href.endswith(".pdf"):

            pdf_url = "https://www.supremecourt.gov" + href
            row = link.find_parent("tr")

            if row:

                cols = row.find_all("td")

                if len(cols) >= 4:

                    date = cols[1].get_text(strip=True)
                    docket = cols[2].get_text(strip=True)
                    case_name = cols[3].get_text(strip=True)
                    justice = cols[4].get_text(strip=True)

                    cases.append({
                        "date": date,
                        "docket": docket,
                        "case_name": case_name,
                        "justice": justice,
                        "pdf_url": pdf_url
                    })

len(cases)
cases


Scraping: https://www.supremecourt.gov/opinions/slipopinion/18
Scraping: https://www.supremecourt.gov/opinions/slipopinion/19
Scraping: https://www.supremecourt.gov/opinions/slipopinion/20
Scraping: https://www.supremecourt.gov/opinions/slipopinion/21
Scraping: https://www.supremecourt.gov/opinions/slipopinion/22
Scraping: https://www.supremecourt.gov/opinions/slipopinion/23


[{'date': '7/14/20',
  'docket': '20A8',
  'case_name': 'Barr v. Lee',
  'justice': 'PC',
  'pdf_url': 'https://www.supremecourt.gov/opinions/19pdf/591us2r63_1bn2.pdf'},
 {'date': '7/09/20',
  'docket': '17-1107',
  'case_name': 'Sharp v. Murphy',
  'justice': 'PC',
  'pdf_url': 'https://www.supremecourt.gov/opinions/19pdf/591us2r62_9o6b.pdf'},
 {'date': '7/09/20',
  'docket': '18-9526',
  'case_name': 'McGirt v. Oklahoma',
  'justice': 'NG',
  'pdf_url': 'https://www.supremecourt.gov/opinions/19pdf/591us2r61_c0nd.pdf'},
 {'date': '7/09/20',
  'docket': '19-715',
  'case_name': 'Trump v. Mazars USA, LLP',
  'justice': 'R',
  'pdf_url': 'https://www.supremecourt.gov/opinions/19pdf/591us2r60_lkgm.pdf'},
 {'date': '7/09/20',
  'docket': '19-635',
  'case_name': 'Trump v. Vance',
  'justice': 'R',
  'pdf_url': 'https://www.supremecourt.gov/opinions/19pdf/591us2r59_g3bi.pdf'},
 {'date': '7/08/20',
  'docket': '19-267',
  'case_name': 'Our Lady of Guadalupe School v. Morrissey-Berru',
  'jus

In [20]:
data = []

for i, case in enumerate(cases):

    pdf_url = case["pdf_url"]

    print(f"Processing {i+1}/{len(cases)}")

    try:

        response = session.get(pdf_url, headers=headers)

        if "application/pdf" not in response.headers.get("Content-Type", ""):
            print("Skipped:", pdf_url)
            continue

        reader = PdfReader(BytesIO(response.content))

        text = ""
        for page in reader.pages:
            t = page.extract_text()
            if t:
                text += t

        word_count = len(text.split())
        page_count = len(reader.pages)
        dissent_count = text.lower().count("dissent")
        concurrence_count = text.lower().count("concur")
        first_amend_count = text.lower().count("first amendment")
        fourth_amend_count = text.lower().count("fourth amendment")
        fifth_amend_count = text.lower().count("fifth amendment")
        sixth_amend_count = text.lower().count("sixth amendment")
        eigth_amend_count = text.lower().count("eighth amendment")

        data.append({
            "date": case["date"],
            "docket": case["docket"],
            "case_name": case["case_name"],
            "justice": case["justice"],
            "pdf_url": pdf_url,
            "word_count": word_count,
            "page_count": page_count,
            "dissent_mentions": dissent_count,
            "concurrence_mentions": concurrence_count,
            "first_amend_mentions": first_amend_count,
            "fourth_amend_mentions": fourth_amend_count,
            "fifth_amend_mentions": fifth_amend_count,
            "sixth_amend_mentions": sixth_amend_count,
            "eigth_amend_mentions": eigth_amend_count
        })

        #time.sleep(1)

    except Exception as e:
        print("Error:", pdf_url)
        print(e)


# STEP 3: Create dataset
df = pd.DataFrame(data)

Processing 1/270
Processing 2/270
Processing 3/270
Processing 4/270
Processing 5/270
Processing 6/270
Processing 7/270
Processing 8/270
Processing 9/270
Processing 10/270
Processing 11/270
Processing 12/270
Processing 13/270
Processing 14/270
Processing 15/270
Processing 16/270
Processing 17/270
Processing 18/270
Processing 19/270
Processing 20/270
Processing 21/270
Processing 22/270
Processing 23/270
Processing 24/270
Processing 25/270
Processing 26/270
Processing 27/270
Processing 28/270
Processing 29/270
Processing 30/270
Processing 31/270
Processing 32/270
Processing 33/270
Processing 34/270
Processing 35/270
Processing 36/270
Processing 37/270
Processing 38/270
Processing 39/270
Processing 40/270
Processing 41/270
Processing 42/270
Processing 43/270
Processing 44/270
Processing 45/270
Processing 46/270
Processing 47/270
Processing 48/270
Processing 49/270
Processing 50/270
Processing 51/270
Processing 52/270
Processing 53/270
Processing 54/270
Processing 55/270
Processing 56/270
P

In [23]:
df.sample(5)

,date,docket,case_name,justice,pdf_url,word_count,page_count,dissent_mentions,concurrence_mentions,first_amend_mentions,fourth_amend_mentions,fifth_amend_mentions,sixth_amend_mentions,eigth_amend_mentions
148,12/10/21,21-463,Whole Woman’s Health v. Jackson,NG,https://www.supremecourt.gov/opinions/21pdf/59...,16458,46,17,15,1,0,0,0,0
209,1/23/23,21-432,Arellano v. McDonough,AB,https://www.supremecourt.gov/opinions/22pdf/59...,4953,16,0,1,0,0,0,0,0
158,6/27/23,21-1168,Mallory v. Norfolk Southern R. Co,NG,https://www.supremecourt.gov/opinions/22pdf/60...,21108,61,46,38,0,0,0,0,0
199,4/18/23,"156, Orig.",New York v. New Jersey,BK,https://www.supremecourt.gov/opinions/22pdf/59...,4067,14,0,1,0,0,0,0,0
106,6/15/22,20-1114,American Hospital Assn. v. Becerra,BK,https://www.supremecourt.gov/opinions/21pdf/59...,5496,18,3,0,0,0,0,0,0


In [25]:
len(df)

df.to_csv("supreme_court_dataset.csv", index=False)
